# Stage 1: MiniMind 白盒拆解與 Pre-training 實戰

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EndeavorYen/LLM-learning/blob/main/notebooks/01_stage1_minimind/01_minimind_pretrain_colab.ipynb)

本 Notebook 帶領你完成第一階段（Stage 1）的核心目標：
1. **環境檢查與 Google Drive 綁定**（防止斷線中斷丟失權重）
2. **下載 MiniMind 專案與輕量語料**
3. **張量維度深入拆解**：剖析 `input_ids`、`logits`、Causal Mask 與 `CrossEntropyLoss` 的底層張量流動
4. **SFT Prompt Loss Masking 原理**：親自理解標籤 `-100` 的作用
5. **啟動 Pre-training 訓練迴圈** 並觀察 Loss 收斂
6. **推論測試**（從隨機亂碼到具備基本字詞流暢度）

## 1. 環境檢查 (GPU & CUDA)

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 掛載 Google Drive (保存 Checkpoint)
> **注意**：Colab 的本地硬碟在 session 斷開後會被完全清空。將訓練 Checkpoint 存於 Google Drive 是必備操作。

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_save_dir = '/content/drive/MyDrive/LLM_learning_checkpoints/stage1_minimind'
    os.makedirs(drive_save_dir, exist_ok=True)
    print(f"Checkpoints will be backed up to: {drive_save_dir}")
except Exception as e:
    print("Running locally or Google Drive not mounted:", e)

## 3. Clone MiniMind 並檢查核心依賴
> **為什麼不直接跑 `pip install -r requirements.txt`？**
> MiniMind 官方的 `requirements.txt` 鎖死了 30 多個舊版套件（如 `numpy==1.26.4`、`scikit_learn==1.5.1`、`sentencepiece==0.2.0`），在 Colab 最新的 Python 3.13 + PyTorch 2.11 環境中會觸發長達數分鐘的 C++ 原始碼重新編譯，並破壞 Colab 內建套件相依性。
> 實際掃描 MiniMind 全部訓練原始碼後，核心訓練只用到 `torch`、`transformers`、`datasets`、`tokenizers`（Colab 均已預裝或可直接相容安裝）。

In [ ]:
import os
import importlib.util

# 1. 下載 MiniMind 專案 (若已下載則直接進入)
if not os.path.exists('/content/minimind'):
    !git clone --filter=blob:none https://github.com/jingyaogong/minimind.git /content/minimind && git -C /content/minimind checkout -q 233d00fa068ec2b8c0d4f07b703934d6391a5a73  # [audit #17] pinned
%cd /content/minimind

# 2. 僅安裝缺失的核心套件，不降級 Colab 原生 numpy / torch / setuptools
required_pkgs = ["datasets", "transformers", "tokenizers"]
missing_pkgs = [pkg for pkg in required_pkgs if importlib.util.find_spec(pkg) is None]
if missing_pkgs:
    pkgs_str = " ".join(missing_pkgs)
    !pip install -q {pkgs_str}

import torch, transformers, datasets
print(f"✅ 環境就緒！PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | Datasets: {datasets.__version__}")

## 4. 深度解析：Pre-training 張量維度變換 (Tensor Shapes)
在啟動訓練前，我們先用一小段原生 PyTorch 代碼看清 Transformer Auto-regressive 的 Loss 計算：

In [ ]:
import torch
import torch.nn as nn

# 模擬 Batch 參數
batch_size = 2
seq_len = 5
vocab_size = 1000

# 模擬模型的輸入與標籤
# 假設文本句子是: [TokenA, TokenB, TokenC, TokenD, TokenE]
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

# 自回歸核心：預測下一個 Token
# 輸入取前 seq_len - 1 個 token
# 目標取第 1 到最後一個 token
inputs = input_ids[:, :-1]   # Shape: [batch_size, seq_len - 1] -> [2, 4]
labels = input_ids[:, 1:]    # Shape: [batch_size, seq_len - 1] -> [2, 4]

# 假設經過 Transformer 後輸出的 logits
logits = torch.randn(batch_size, seq_len - 1, vocab_size) # Shape: [2, 4, 1000]

# CrossEntropyLoss 期望的輸入維度: [N, C] 與目標 [N]
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
loss = loss_fn(logits.reshape(-1, vocab_size), labels.reshape(-1))

print(f"Inputs shape: {inputs.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Logits shape: {logits.shape}")
print(f"Flattened logits shape: {logits.reshape(-1, vocab_size).shape}")
print(f"CrossEntropy Loss: {loss.item():.4f}")

## 5. 深度解析：SFT 中的 Loss Masking (-100)
為什麼 SFT 不能計算整個句子的 Loss？
- 如果對整個句子（Prompt + Answer）都算 Loss，模型會浪費參數量在「背誦使用者的提問」。
- PyTorch 的 `CrossEntropyLoss` 預設 `ignore_index=-100`。
- 我們只要將 Prompt 區間的 label 替換為 `-100`，計算梯度時就會完全略過 Prompt，只計算 Answer 的損失！

In [ ]:
# 範例演示：Prompt 區域打上 -100
# 句子: [Prompt_tok1, Prompt_tok2, Answer_tok1, Answer_tok2]
prompt_len = 2
seq = torch.tensor([[101, 205, 308, 409]]) # shape: [1, 4]

input_tokens = seq[:, :-1] # [101, 205, 308]
target_tokens = seq[:, 1:].clone() # [205, 308, 409]

# 將 Prompt 區間在 target 設為 -100
# 第 0 個位置預測 205 (Prompt 結尾)，第 1 個位置預測 308 (Answer 開始)
target_tokens[:, :prompt_len-1] = -100 

print(f"Input tokens:  {input_tokens.tolist()[0]}")
print(f"Target labels: {target_tokens.tolist()[0]}  (-100 代表不計算損失)")

## 6. 準備預訓練資料集並啟動 MiniMind Pre-training
MiniMind 的核心訓練腳本位於 `trainer/train_pretrain.py`。
下面會下載官方輕量預訓練資料集 `pretrain_t2t_mini.jsonl`，並額外抽取前 **10,000 筆** 作為快速驗證集 `pretrain_fast_10k.jsonl`（約 1~2 分鐘即可跑完並驗證收斂；若想跑完整 1.2GB 語料，可改用 `--data_path ../dataset/pretrain_t2t_mini.jsonl`）。

In [ ]:
import os
os.makedirs('/content/minimind/dataset', exist_ok=True)
full_data_path = '/content/minimind/dataset/pretrain_t2t_mini.jsonl'
fast_data_path = '/content/minimind/dataset/pretrain_fast_10k.jsonl'

if not os.path.exists(full_data_path):
    print("正在從 Hugging Face 下載 MiniMind 預訓練數據集 (約 1 分鐘)...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="jingyaogong/minimind_dataset",
        filename="pretrain_t2t_mini.jsonl",
        repo_type="dataset",
        local_dir="/content/minimind/dataset"
    )

if not os.path.exists(fast_data_path):
    !head -n 10000 {full_data_path} > {fast_data_path}

print(f"✅ 數據集已就緒！\n - 快速驗證集 (10,000 筆): {fast_data_path}\n - 完整輕量集: {full_data_path}")

### 啟動 Pre-training 訓練迴圈 (26M 模型: `hidden_size=512, num_hidden_layers=8`)
切換至 `trainer/` 目錄並執行預訓練腳本（觀察 Loss 從 `~8.8` 逐步下降）：

In [ ]:
%cd /content/minimind/trainer
# 使用 10,000 筆快速驗證集跑 1 個 epoch (約 313 steps，T4/L4 約 1~2 分鐘完成)
# 若想跑完整語料，將 --data_path 改為 ../dataset/pretrain_t2t_mini.jsonl 即可
!python train_pretrain.py \
    --data_path ../dataset/pretrain_fast_10k.jsonl \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --epochs 1 \
    --batch_size 32 \
    --accumulation_steps 2 \
    --learning_rate 5e-4 \
    --log_interval 20 \
    --save_interval 100

## 7. 備份權重至 Google Drive

In [ ]:
# 將訓練產出的 out/pretrain_512.pth 同步到 Google Drive
!cp -r /content/minimind/out/* /content/drive/MyDrive/LLM_learning_checkpoints/stage1_minimind/ 2>/dev/null || echo "Out dir not created or Drive not available"
!ls -lh /content/minimind/out/

## 8. 推論測試 (Text Generation)
使用根目錄下的 `eval_llm.py` 載入剛訓練出的 `out/pretrain_512.pth` 進行續寫測試：

In [ ]:
%cd /content/minimind
# 自動輸入 0 (選擇自動測試題庫)，載入 pretrain_512.pth
!echo 0 | python eval_llm.py --weight pretrain --hidden_size 512 --num_hidden_layers 8 --max_new_tokens 64